## Precipitation from CHIRPS Daily (Google Earth Engine)

This notebook fetches **UCSB-CHG/CHIRPS/DAILY** precipitation and, for each sample in **water_quality_training_dataset.csv** (Latitude, Longitude, Sample Date), derives:

1. **Sum of precipitation in the 7 days before** the sampling date (days D-7 through D-1, mm).
2. **Number of days since last rain** (days since the most recent day with precipitation above a threshold, e.g. 0.1 mm).

### Prerequisites

1. **Install** the Earth Engine API: `pip install earthengine-api`
2. **Authenticate** (one-time): run the cell with `ee.Authenticate()`, then `ee.Initialize()`

### Dataset

[CHIRPS Daily](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY): daily precipitation (mm/day), 0.05° resolution (~5566 m), 1981–present, band `precipitation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

import ee

### 1. Authenticate and initialize (run once)

Uncomment and run the authenticate cell once if needed, then run initialize.

In [ ]:
# One-time: opens browser to sign in with Google and grant Earth Engine access
# ee.Authenticate()

True

In [4]:
try:
    ee.Initialize()
    print("Earth Engine initialized.")
except Exception as e:
    print("Run ee.Authenticate() first, then ee.Initialize(). Error:", e)

Earth Engine initialized.


### 2. Load water quality training data (Latitude, Longitude, Sample Date)

We use **data/original/water_quality_training_dataset.csv**. Sample Date format is DD-MM-YYYY.

In [26]:
# Paths
if os.path.exists("data"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
ORIGINAL_DIR = os.path.join(DATA_DIR, "original")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

train_path = os.path.join(ORIGINAL_DIR, "water_quality_training_dataset.csv")
df = pd.read_csv(train_path)

# Parse Sample Date (DD-MM-YYYY)
df["sample_date"] = pd.to_datetime(df["Sample Date"], format="%d-%m-%Y")
print(f"Rows: {len(df)}, date range: {df['sample_date'].min()} to {df['sample_date'].max()}")

Rows: 9319, date range: 2011-01-02 00:00:00 to 2015-12-31 00:00:00


### 3. Parameters

CHIRPS scale ~5566 m. "7 days before" = D-7 through D-1 (inclusive). "Days since last rain" uses a rain threshold (mm) and a lookback window (e.g. 90 days).

In [6]:
CHIRPS_SCALE = 5566  # meters
DAYS_BEFORE_FOR_SUM = 7   # sum of precip in 7 days before sample date
LOOKBACK_DAYS = 90       # look back up to 90 days for "days since last rain"
RAIN_THRESHOLD_MM = 0.1  # day counts as "rain" if precip >= this (mm)

### 4. Process by unique sample date

For each unique **sample_date**:
1. **7-day sum**: Filter CHIRPS for (date - 7, date), sum to one image, sample at all points with that date.
2. **Days since last rain**: Filter CHIRPS for (date - lookback, date), get daily time series at all points via `getRegion`, then in Python compute days since last rain per point.

**How `get_precip_7d_sum_and_days_since_rain` works**

- **Input**: DataFrame with `Latitude`, `Longitude`, `Sample Date`, and a parsed `sample_date` (datetime).
- **Grouping**: Rows are grouped by unique `sample_date` so we do one GEE request per date (not per row).
- **7-day sum (mm)**:
  - For each date, CHIRPS is filtered to `[sample_date - days_before, sample_date)` (GEE end-exclusive), so that’s D-7 through D-1.
  - The collection is summed to a single image; that image is sampled at each (lon, lat) with `reduceRegions` at `chirps_scale` (~5566 m). The sampled value is the 7-day precipitation sum in mm.
- **Days since last rain**:
  - CHIRPS is filtered to `[sample_date - lookback_days, sample_date)` and `getRegion` returns the daily time series at all points (one row per point per day).
  - In Python, for each point we sort days newest-first and find the first day with precipitation ≥ `rain_threshold_mm` (0.1 mm). The index of that day (0 = yesterday, 1 = 2 days ago, …) is `days_since_last_rain`. If there’s no rain in the lookback, we set it to `lookback_days`.
- **Output**: DataFrame with original location/date columns plus `precip_7d_sum_mm` and `days_since_last_rain` (one row per input row). Results can be NaN if CHIRPS has no data for that date/place or if a GEE call fails.

The loop above uses row index in `sub` for matching; we need to match by position (order in `sub`) for `precip_7d_by_id`. Fixing the indexing: `id` in the FeatureCollection is `i` from `enumerate(coords)`, so the k-th row in `sub` has `id = k`. So we use `precip_7d_by_id.get(i, np.nan)` where `i` is the row index in the loop. Actually we're iterating `for i, row in sub.iterrows()` so `i` is the original dataframe index, not 0,1,2. So we should enumerate(sub) to get 0,1,2 for id. Let me fix the code to use explicit index 0..len(sub)-1 for id and then when building results we iterate with enumerate(sub) and use j as id.

In [ ]:
def get_precip_7d_sum_and_days_since_rain(df_train, chirps_scale=5566, days_before=7, rain_threshold_mm=0.1):
    """
    For each row in df_train (must have Latitude, Longitude, sample_date):
    - precip_7d_sum_mm: sum of CHIRPS precipitation in the 7 days before sample_date (mm)
    - days_since_last_rain: number of days since most recent day with precip >= rain_threshold_mm
    """
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
    results = []
    unique_dates = df_train["sample_date"].drop_duplicates().sort_values()

    for sample_date in unique_dates:
        sample_date = pd.Timestamp(sample_date)
        start_7d = (sample_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
        end_date = sample_date.strftime("%Y-%m-%d")

        mask = df_train["sample_date"] == sample_date
        sub = df_train.loc[mask].reset_index(drop=True)
        if sub.empty:
            continue

        lats = sub["Latitude"].tolist()
        lons = sub["Longitude"].tolist()
        coords = [[lon, lat] for lon, lat in zip(lons, lats)]

        # 1) 7-day sum (GEE filterDate end is exclusive: [start_7d, end_date) = D-7..D-1)
        chirps_7d_coll = chirps.filterDate(start_7d, end_date)
        precip_7d_by_id = {j: np.nan for j in range(len(coords))}
        n_images = chirps_7d_coll.size().getInfo()
        if n_images and n_images > 0:
            chirps_7d = chirps_7d_coll.sum()
            points_fc = ee.FeatureCollection([
                ee.Feature(ee.Geometry.Point(c), {"id": j})
                for j, c in enumerate(coords)
            ])
            reduced_7d = chirps_7d.reduceRegions(
                collection=points_fc,
                reducer=ee.Reducer.first(),
                scale=chirps_scale,
            )
            info_7d = reduced_7d.getInfo()
            for f in info_7d.get("features", []):
                props = f.get("properties", {})
                idx = props.get("id")
                p = props.get("precipitation")
                if p is None:
                    for k, v in props.items():
                        if k != "id" and v is not None:
                            try:
                                p = float(v)
                                break
                            except (TypeError, ValueError):
                                pass
                precip_7d_by_id[idx] = float(p) if p is not None else np.nan

        
        for j, (_, row) in enumerate(sub.iterrows()):
            results.append({
                "Latitude": row["Latitude"],
                "Longitude": row["Longitude"],
                "Sample Date": row["Sample Date"],
                "sample_date": sample_date,
                "precip_7d_sum_mm": precip_7d_by_id.get(j, np.nan),
            })
    return pd.DataFrame(results)

In [27]:
# Run the extraction (this may take a while: one reduceRegions + one getRegion per unique sample date)
from tqdm import tqdm

unique_dates = df["sample_date"].drop_duplicates()
df = df.loc[0:10]
print(f"Unique sample dates: {len(unique_dates)}")

precip_df = get_precip_7d_sum_and_days_since_rain(
    df,
    chirps_scale=CHIRPS_SCALE,
    days_before=DAYS_BEFORE_FOR_SUM,
    lookback_days=LOOKBACK_DAYS,
    rain_threshold_mm=RAIN_THRESHOLD_MM
)
print(f"Result rows: {len(precip_df)}")
precip_df.head(10)

Unique sample dates: 1364
Result rows: 11


,Latitude,Longitude,Sample Date,sample_date,precip_7d_sum_mm,days_since_last_rain
0,-28.760833,17.730278,02-01-2011,2011-01-02,0.000000,NaN
1,-26.861111,28.884722,03-01-2011,2011-01-03,47.329471,NaN
2,-26.450000,28.085833,03-01-2011,2011-01-03,34.640530,NaN
3,-27.671111,27.236944,03-01-2011,2011-01-03,72.844851,NaN
4,-27.356667,27.286389,03-01-2011,2011-01-03,87.171568,NaN
5,-27.010111,26.698083,04-01-2011,2011-01-04,25.602541,NaN
6,-25.127778,27.628889,04-01-2011,2011-01-04,55.679502,NaN
7,-25.206390,27.558000,04-01-2011,2011-01-04,35.628225,NaN
8,-24.695140,27.409060,04-01-2011,2011-01-04,26.471423,NaN
9,-26.984722,26.632278,04-01-2011,2011-01-04,24.788070,NaN


In [10]:
# Merge back to training dataframe on (Latitude, Longitude, Sample Date)
df_merged = df.merge(
    precip_df[["Latitude", "Longitude", "Sample Date", "precip_7d_sum_mm", "days_since_last_rain"]],
    on=["Latitude", "Longitude", "Sample Date"],
    how="left",
)
df_merged.head(10)

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,sample_date,precip_7d_sum_mm,days_since_last_rain
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,2011-01-02,NaN,NaN
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,2011-01-03,NaN,NaN
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,2011-01-03,NaN,NaN
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,2011-01-03,NaN,NaN
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,2011-01-03,NaN,NaN
5,-27.010111,26.698083,04-01-2011,82.200,289.8,192.0,2011-01-04,NaN,NaN
6,-25.127778,27.628889,04-01-2011,125.000,438.0,163.0,2011-01-04,NaN,NaN
7,-25.206390,27.558000,04-01-2011,116.620,568.0,69.0,2011-01-04,NaN,NaN
8,-24.695140,27.409060,04-01-2011,181.831,583.0,158.0,2011-01-04,NaN,NaN
9,-26.984722,26.632278,04-01-2011,196.000,452.0,158.0,2011-01-04,NaN,NaN


In [11]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = os.path.join(PROCESSED_DIR, "precipitation_chirps_training.csv")
precip_df.to_csv(out_path, index=False)
print(f"Saved {len(precip_df)} rows to {out_path}")

Saved 9319 rows to /Users/ben/Documents/Projects/ey_challenge_2026/data/processed/precipitation_chirps_training.csv


### Notes

- **7 days before**: CHIRPS images for (sample_date - 7) through (sample_date - 1) are summed; sample date itself is excluded.
- **Days since last rain**: We look back up to `LOOKBACK_DAYS` (90); a day counts as "rain" if precipitation ≥ `RAIN_THRESHOLD_MM` (0.1 mm). If no rain in the lookback, we set days_since_last_rain = lookback_days.
- **Performance**: One `reduceRegions` (getInfo) and one `getRegion` (getInfo) per unique sample date. For many unique dates, consider batching or exporting from GEE.
- **getRegion**: For MultiPoint, GEE returns one row per (point, image). Column order may vary; we detect "longitude", "latitude", "time", "precipitation" from the header row.